# task 1 data cleaning

This notebook cleans the `Region summary_ New South Wales STE 1.csv` file for task 1. The main idea is to keep the missing values carefully, because many indicators are only available in selected years, and then reshape the dataset into a long format for later analysis.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

current_dir = Path.cwd()
file_path = current_dir / "data" / "Region summary_ New South Wales STE 1.csv"

df = pd.read_csv(file_path)
display(df.head())


## basic structure

The original file is a wide table. Each row is one indicator, and the year values are spread across many columns.

In [ ]:
structure_df = pd.DataFrame(
    {
        "item": ["rows", "columns", "duplicate_rows"],
        "value": [len(df), len(df.columns), int(df.duplicated().sum())],
    }
)

display(structure_df)
display(pd.DataFrame({"column_name": df.columns}))


## cleaning steps

The cleaning below does four things:

1. standardises column names
2. extracts the unit from the description column
3. counts how many years each indicator actually has
4. reshapes the table from wide format into long format

In [ ]:
task1_df = df.copy()

task1_df = task1_df.rename(
    columns={
        "Measure Code": "measure_code",
        "Parent Description": "parent_description",
        "Description": "description",
    }
)

task1_df.columns = [f"y_{col}" if str(col).isdigit() else col for col in task1_df.columns]
year_cols = [col for col in task1_df.columns if col.startswith("y_")]

task1_df["unit"] = task1_df["description"].str.extract(r"\(([^()]*)\)\s*$")
task1_df["description_clean"] = task1_df["description"].str.replace(r"\s*\([^()]*\)\s*$", "", regex=True)
task1_df["non_null_years"] = task1_df[year_cols].notna().sum(axis=1)

task1_df = task1_df[task1_df["non_null_years"] > 0].copy()

display(task1_df.head())


## long format

For analysis, the cleaned table is converted into a long table with one row per indicator-year pair.

In [ ]:
task1_long_df = task1_df.melt(
    id_vars=[
        "measure_code",
        "parent_description",
        "description",
        "description_clean",
        "unit",
        "non_null_years",
    ],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

task1_long_df = task1_long_df.dropna(subset=["value"]).copy()
task1_long_df["year"] = task1_long_df["year"].str.replace("y_", "").astype(int)

display(task1_long_df.head())


## analysis table

The year 2025 only has three non-null pension records, so it is kept in the cleaned long table but removed from the main analysis table.

In [ ]:
task1_analysis_df = task1_long_df[task1_long_df["year"] != 2025].copy()

cleaning_summary_df = pd.DataFrame(
    {
        "item": [
            "original_rows",
            "rows_after_drop_all_null_years",
            "long_table_rows",
            "analysis_rows_without_2025",
            "rows_with_only_one_non_null_year",
            "non_null_values_in_2025",
        ],
        "value": [
            len(df),
            len(task1_df),
            len(task1_long_df),
            len(task1_analysis_df),
            int((task1_df["non_null_years"] == 1).sum()),
            int(task1_long_df[task1_long_df["year"] == 2025].shape[0]),
        ],
    }
)

display(cleaning_summary_df)
display(task1_analysis_df.head())
